# Academic Credential Verifier

Verify student credentials against a Neo4j graph database using Gemini AI for PDF parsing.

## 1. Install Dependencies

In [1]:
!pip install google-generativeai

In [5]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.4 MB/s eta 0:00:00


In [18]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 62.9 MB/s eta 0:00:00


## 2. Configure Gemini AI

In [ ]:
import getpass
import google.generativeai as genai

API_KEY = getpass.getpass("Enter Gemini API Key: ")

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini model ready.")

## 3. Connect to Neo4j

In [ ]:
from neo4j import GraphDatabase

URI = input("Enter Aura URI: ")
USERNAME = input("Enter Aura Username: ")
PASSWORD = getpass.getpass("Enter Aura Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()
print("Connected!")

In [7]:
with driver.session() as session:

    result = session.run("""
    MATCH (s:Student)
    RETURN s.name AS name
    """)

    for record in result:
        print(record["name"])

Bhargavi
Rahul
Aditi


## 4. Define Verification Functions

In [25]:
def verify_credential_detailed(name, university, degree, year):

    query = """
    MATCH (s:Student {name:$name})
          -[:STUDIED_AT]->(u:University),
          (s)-[:HAS_DEGREE]->(d:Degree),
          (s)-[:GRADUATED_IN]->(y:Year)

    RETURN
        u.name AS university,
        d.name AS degree,
        y.value AS year
    """

    with driver.session() as session:

        result = session.run(query, name=name)
        record = result.single()

        if record is None:
            return {
                "status": "NOT VERIFIED",
                "reason": "Student not found"
            }

        failures = []

        if record["university"] != university:
            failures.append("University mismatch")

        if record["degree"] != degree:
            failures.append("Degree mismatch")

        if record["year"] != year:
            failures.append("Year mismatch")

        if len(failures) == 0:
            return {
                "status": "VERIFIED",
                "reason": "All fields matched"
            }

        return {
            "status": "NOT VERIFIED",
            "reason": ", ".join(failures)
        }

In [ ]:
def generate_report(credential, result):

    report = f"""
Academic Credential Verification Report

Student: {credential['name']}
University: {credential['university']}
Degree: {credential['degree']}
Year: {credential['year']}

Status: {result['status']}
Reason: {result['reason']}
"""

    return report

## 5. Seed Sample Data

Run this once to populate the Neo4j database with sample student records for testing.

In [ ]:
def seed_sample_data():
    cypher = """
    // Student 1: correct record — should VERIFY
    MERGE (s1:Student {name: "Bhargavi"})
    MERGE (u1:University {name: "Manipal University Jaipur"})
    MERGE (d1:Degree {name: "B.Tech CSE"})
    MERGE (y1:Year {value: 2028})
    MERGE (s1)-[:STUDIED_AT]->(u1)
    MERGE (s1)-[:HAS_DEGREE]->(d1)
    MERGE (s1)-[:GRADUATED_IN]->(y1)

    // Student 2: wrong degree and year — should NOT VERIFY
    MERGE (s2:Student {name: "Prerna"})
    MERGE (u2:University {name: "VIT Vellore"})
    MERGE (d2:Degree {name: "B.Tech ECE"})
    MERGE (y2:Year {value: 2026})
    MERGE (s2)-[:STUDIED_AT]->(u2)
    MERGE (s2)-[:HAS_DEGREE]->(d2)
    MERGE (s2)-[:GRADUATED_IN]->(y2)
    """
    with driver.session() as session:
        session.run(cypher)
    print("Sample data seeded successfully.")

seed_sample_data()

## 6. Upload and Parse Certificate PDF

In [17]:
from google.colab import files

uploaded = files.upload()

Saving Certificate1_Version control with git and github.pdf to Certificate1_Version control with git and github.pdf


In [ ]:
import fitz

pdf_file = list(uploaded.keys())[0]

doc = fitz.open(pdf_file)

text = ""

for page in doc:
    text += page.get_text()

print(text[:3000])

## 7. Extract Credentials with Gemini AI

In [ ]:
prompt = f"""
Extract the following information from the certificate.

Return ONLY valid JSON.

Fields:
name
university
degree
year

Certificate:
{text}
"""

response = model.generate_content(prompt)

print(response.text)

In [ ]:
import json

raw = response.text.strip()

# Strip markdown code fences if present
if raw.startswith("```"):
    raw = raw.split("\n", 1)[-1]
    raw = raw.rsplit("```", 1)[0]

try:
    credential = json.loads(raw.strip())
    print("Extracted credential:", credential)
except json.JSONDecodeError as e:
    print("Failed to parse JSON. Raw response was:")
    print(raw)
    raise e

## 8. Verify Extracted Credentials

In [ ]:
verification_result = verify_credential_detailed(
    credential["name"],
    credential["university"],
    credential["degree"],
    credential["year"]
)

print("Verification Result:", verification_result)

## 9. Generate Verification Report

In [ ]:
print(generate_report(credential, verification_result))

## 10. Explanation

In [ ]:
import json

prompt = f"""
Explain the following academic credential verification result in clear, simple language.

Credential submitted:
{json.dumps(credential, indent=2)}

Verification result:
Status: {verification_result["status"]}
Reason: {verification_result["reason"]}
"""

response = model.generate_content(prompt)

print(response.text)